# Attention Sinks
### *Efficient Streaming Language Models with Attention Sinks* — Xiao et al. 2023

**Paper:** [arXiv:2309.17453](https://arxiv.org/abs/2309.17453)

---

## The big picture

When you peek inside a trained language model's attention weights, you find something
surprising: **the very first token** in the sequence — even if it's just punctuation or
a BOS marker — consistently absorbs a huge fraction of the total attention mass across
*every* head and *every* layer.

The first token is called an **attention sink**.

This notebook will:
1. Train a small GPT on synthetic text
2. Measure the sink effect quantitatively
3. Show why removing the sink token breaks streaming inference
4. Verify the fix proposed by Xiao et al.


## Setup

Install dependencies if needed:
```
pip install torch numpy matplotlib tqdm
```


In [ ]:
import sys, os
# Add repo root to path (works whether running from notebooks/ or root)
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from mineo.experiments.attention_sinks import AttentionSinkExperiment
from mineo.visualization.plots import (
    plot_sink_scores, plot_cache_comparison, plot_training_loss,
    plot_attention_heatmap,
)

print(f"PyTorch {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")


## Theory: Why does an attention sink form?

### The softmax constraint

Attention weights at each query position must sum to 1 (softmax normalisation).
When the model wants to "not attend to anything in particular", it still needs
somewhere to put the surplus weight.

### Why position 0?

The causal mask means every query can only attend to past tokens.  Position 0 is
the *only* token reachable from **every** query position.  So it becomes the model's
universal dump site — a token that's always available as an overflow receptacle.

Over training, this pattern gets reinforced: since the model uses position 0 as a
sink, it stops encoding useful information there, which makes it an even better sink.

### The streaming cache problem

![Streaming cache diagram](https://i.imgur.com/placeholder.png)

During streaming inference, a **sliding-window KV cache** keeps only the last $W$
tokens to bound memory.  Once position 0 slides out of the window:

- The model's attention can no longer access the sink
- The softmax has no natural dump site
- Attention weights become erratic → perplexity spikes

**Fix (Xiao et al.):** Always keep the first $k$ tokens ("sink tokens") in the cache
alongside the sliding window.


## Step 1: Train a small GPT


In [ ]:
exp = AttentionSinkExperiment(
    d_model    = 128,   # embedding dimension
    n_layers   = 4,     # transformer depth
    n_heads    = 4,     # attention heads
    seq_len    = 128,   # context length
    n_steps    = 2_000, # training steps (~2 min on CPU)
    batch_size = 32,
    device     = device,
    verbose    = True,
)

print(f"Model parameters: {exp.model.n_params:,}")


In [ ]:
losses = exp.train(log_interval=400)
fig = plot_training_loss(losses, title='Attention Sink Model — Training Loss')
plt.show()


## Step 2: Measure the attention sink

For each sample, we compute how much **total attention mass** each token position
receives (summed over all query positions, averaged over heads and layers).

Under a uniform distribution this would be $1/T$ at every position.  We look for
position 0 to be a significant outlier.


In [ ]:
sink_stats = exp.measure_sink_scores(n_samples=64)

print(f"Sink ratio (pos-0 vs. uniform): {sink_stats['sink_ratio']:.2f}×")
print()
print("Attention mass at the first 10 positions:")
for i, v in enumerate(sink_stats['mean_attn_by_pos'][:10].tolist()):
    bar = '█' * int(v * 200)
    print(f"  pos {i:3d}: {v:.4f}  {bar}")


In [ ]:
fig = plot_sink_scores(
    sink_stats['mean_attn_by_pos'],
    sink_stats['per_layer'],
    title=f"Attention mass per token position  (sink ratio: {sink_stats['sink_ratio']:.1f}×)",
)
plt.show()


### What you should see

- Position 0 (red bar) has dramatically more attention mass than any other position.
- The effect persists **across all layers** (bottom panel), not just early or late ones.
- This is the attention sink.

Try varying `n_heads` and `n_layers` — the sink appears regardless of architecture size.


## Step 3: The streaming cache experiment

We now simulate three KV-cache strategies for streaming inference:

| Strategy | What it keeps |
|---|---|
| `full_context` | All tokens — memory grows with sequence length |
| `window_only` | Last $W$ tokens — **evicts the sink token** |
| `sink_plus_window` | Sink token(s) + last $W-1$ tokens — Xiao et al. fix |

We measure **perplexity** (lower = better) for each strategy.


In [ ]:
cache_stats = exp.simulate_streaming_cache(cache_size=32, n_eval_tokens=256)

print("\nPerplexity comparison (lower = better):")
for strategy, ppl in cache_stats.items():
    indicator = ' ←  BEST' if ppl == min(cache_stats.values()) else ''
    print(f"  {strategy:<22s}: {ppl:.2f}{indicator}")


In [ ]:
fig = plot_cache_comparison(cache_stats, title='Streaming KV-cache: perplexity comparison')
plt.show()


### Result

`sink_plus_window` should match `full_context` closely, while `window_only`
(which loses the sink token) has higher perplexity.

This directly validates the Xiao et al. fix: **keeping just the sink token(s)**
is enough to recover full-context performance.

---

## Visualising a single attention matrix

Let's look at what the attention pattern actually looks like for one sequence.


In [ ]:
exp.model.eval()
x, _ = exp.dataset.get_batch(1, torch.device(device))
_, _, attn_list = exp.model(x, return_attn=True)

# Show layer 0, head 0
attn_layer0 = attn_list[0][0, 0].cpu()  # (T, T)

fig, axes = plt.subplots(1, len(attn_list), figsize=(4 * len(attn_list), 4))
for i, attn in enumerate(attn_list):
    a = attn[0].mean(0).cpu().numpy()  # average over heads
    axes[i].imshow(a, cmap='Blues', aspect='auto')
    axes[i].set_title(f'Layer {i}\n(avg over heads)')
    axes[i].set_xlabel('Key pos')
    if i == 0:
        axes[i].set_ylabel('Query pos')

plt.suptitle('Attention weights: brighter = more attention\n'
             'Notice the bright left column — the attention sink at pos 0', y=1.02)
plt.tight_layout()
plt.show()


## Summary

| Observation | Explanation |
|---|---|
| Position 0 gets high attention | Softmax needs a dump site; pos 0 is always reachable |
| Effect appears in all layers | The pattern is reinforced during training |
| Removing sink → high perplexity | Model relies on the sink being present |
| Keeping 1–4 sink tokens fixes it | Cheap, constant-memory solution |

**Key paper takeaway:** When designing long-context or streaming inference systems,
always reserve a few slots for "sink tokens" in your KV cache.

### Further reading
- [StreamingLLM repo](https://github.com/mit-han-lab/streaming-llm)
- Xiao et al. 2023 — [arXiv:2309.17453](https://arxiv.org/abs/2309.17453)
- "Lost in the Middle" (Liu et al. 2023) — related work on positional bias
